# Überprüfung der durchgeführten Anpassungen, um SR27-3 abzubilden

In [0]:
import pandas as pd
import pypsa

BASE = "/Volumes/playground/u_luisa_kamp/pypsa-de_data/SR27-3/results"
RUNS = {"original": f"{BASE}/original", "modified": f"{BASE}/modified"}
YEARS = [2040, 2045]
FILE = "base_s_27__none_{year}.nc"   # Dateiname ggf. anpassen

# Carrier, die uns interessieren (Teilstring-Suche, Groß-/Kleinschreibung egal)
PATTERN = "OCGT|CCGT|CHP|H2|Electrolysis"

In [0]:
nets = {
    (run, year): pypsa.Network(f"{path}/{FILE.format(year=year)}")
    for run, path in RUNS.items()
    for year in YEARS
}

In [0]:
def de_links(n):
    l = n.links[n.links.bus0.str.startswith("DE")].copy()
    l["GW_bus0"] = l.p_nom_opt / 1e3            # Leistung auf bus0-Seite
    l["GW_el"] = l.GW_bus0 * l.efficiency       # nur sinnvoll bei Kraftwerken
    return l

rows = []
for (run, year), n in nets.items():
    l = de_links(n)
    g = l.groupby("carrier").agg(GW_bus0=("GW_bus0", "sum"),
                                 GW_el=("GW_el", "sum"),
                                 eff=("efficiency", "mean"))
    g = g[g.index.str.contains(PATTERN, case=False)]
    g["run"], g["year"] = run, year
    rows.append(g.reset_index())

caps = pd.concat(rows)
caps.pivot_table(index="carrier", columns=["year", "run"],
                 values=["GW_bus0", "GW_el"]).round(2)

urban central H2 CHP fällt von 3,66 auf 0,46 GWel, obwohl die Elektrolyse steigt. Das sollten Sie sich ansehen (H₂-Bilanz, Importmengen). Eine Erklärung habe ich nicht

In [0]:
for year in YEARS:
    print(f"--- {year} ---")
    for run in RUNS:
        l = de_links(nets[(run, year)])
        elz = l[l.carrier == "H2 Electrolysis"].GW_bus0.sum()
        gas = l[l.carrier.str.contains("OCGT|CCGT", case=False)
                & ~l.carrier.str.contains("H2", case=False)].GW_bus0.sum()
        print(f"{run:9s} Elektrolyse: {elz:6.1f} GW (Soll >= 17) | "
              f"Gasturbinen (bus0): {gas:6.1f} GW")

Die Elektrolyse-Untergrenze greift. Im Original baut das Modell nur 2,2 GW, in SR27_S3 genau 17,0 GW in 2040 und 2045. Die Untergrenze ist also bindend. Die Zielfunktion ändert sich kaum (2040: 266,44 → 266,88, 2045: 102,79 → 102,78). Die Untergrenze verzerrt das System damit nicht stark. Die Werte von 2040 und 2045 sind nicht direkt vergleichbar, nur Original gegen Modified im selben Jahr.

In [0]:
pd.Series({k: n.objective / 1e9 for k, n in nets.items()}).round(2)

In [0]:
n = nets[("modified", 2045)]
l = n.links.copy()

l["bus1_carrier"] = l.bus1.map(n.buses.carrier)

mask = (
    l.carrier.str.contains("OCGT|CCGT|CHP|Fuel Cell", case=False, na=False)
    & l.bus1.str.startswith("DE", na=False)
)

cols = [
    "carrier", "bus0", "bus1", "bus1_carrier",
    "bus2", "efficiency", "efficiency2", "p_nom_opt"
]

display(
    l.loc[mask, [c for c in cols if c in l.columns]]
     .drop_duplicates()
)

In [0]:
gas_carriers = [
    "OCGT", "CCGT",
    "urban central gas CHP",
    "urban central gas CHP CC",
]

for year in YEARS:
    n = nets[("modified", year)]
    l = n.links

    mask = (
        l.carrier.isin(gas_carriers)
        & l.bus1.str.startswith("DE", na=False)
        & l.bus1.map(n.buses.carrier).eq("AC")
        & ~l.p_nom_extendable
        & l.active
    )

    fixed_gw_el = (
        l.loc[mask, "p_nom"] * l.loc[mask, "efficiency"]
    ).sum() / 1e3

    target = {2040: 50, 2045: 19}[year]
    print(
        f"{year}: fester Methanbestand {fixed_gw_el:.2f} GW_el"
        f" | Ziel {target} GW_el"
    )

Wir brauchen eine eigene Summenbedingung. Sie begrenzt einzelne Carrier anhand ihrer Eingangsleistung, ohne Wirkungsgrade. Außerdem: Übersteigt der feste Bestand eine Höchstgrenze, verhindert sie lediglich weiteren Ausbau, reduziert aber nicht den Bestand. Danach wissen wir, ob eine zusätzliche Kapazitätsbedingung genügt oder zuerst der feste Bestand angepasst werden muss.

Prüfen, ob der feste Methankraftwerksbestand bereits über deinen Zielwerten liegt. Sonst könnte eine exakte Summenvorgabe unlösbar werden.

In [0]:
display(
    l.loc[mask]
    .assign(GW_el=lambda x: x.p_nom * x.efficiency / 1e3)
    .groupby("carrier")["GW_el"]
    .sum()
    .round(2)
)

Der Überschuss entsteht erst durch die Einbeziehung der Gas-KWK: CCGT und OCGT liegen zusammen unter 19 GW, mit Gas-KWK darüber.